## 3Di alignment baseline establishment

* Read BFVD and produce 3Di frequency matrix
* For each MetaVR protein, generate baseline 3Di sequence based on the frequency matrix
* Calculate pairwise 3Di identity

### Generate 3Di frequency matrix from BFVD

In [2]:
import sys

BFVD_3DI_FA="/work/FAC/FBM/DBC/cdessim2/default/dkim5/db/bfvd/bfvd_3di.fa"

# Read BFVD 3Di sequences and compute frequency matrix
# Ex) A: 123, C: 789, ..., TOTAL: 9999
freq_matrix = {}
with open(BFVD_3DI_FA, "r") as f:
    for line in f:
        if line.startswith(">"):
            continue
        seq = line.strip()
        for c in seq:
            if c not in freq_matrix:
                freq_matrix[c] = 0
            freq_matrix[c] += 1

# Write frequency matrix to file
FREQ_MATRIX_FILE = "bfvd_3di_freq_matrix.tsv"
with open(FREQ_MATRIX_FILE, "w") as f:
    for c, count in freq_matrix.items():
        f.write(f"{c}\t{count}\n")
    f.write(f"TOTAL\t{sum(freq_matrix.values())}\n")

### Protein sequence generator based on frequency matrix

In [4]:
def generate_protein_sequence(freq_matrix, length):
    import random
    total_count = sum(freq_matrix.values())
    chars = list(freq_matrix.keys())
    weights = [freq_matrix[c] / total_count for c in chars]
    return ''.join(random.choices(chars, weights=weights, k=length))

print(generate_protein_sequence(freq_matrix, 100))  # Example usage

DDVDDDVTDVVNDSTCVDVDQDPLDVNPELHVPPAVGVGDLPVAVPPKPLDVPVSVPWDDSDRTWPDVVAMAPYDPPPCLPLRQQSVDDSVTRVDPVVDV


### Generate baseline 3Di sequences for MetaVR proteins

In [5]:
METAVR_3DI_FA="/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/metavr/db/metavr_3di.fa"
OUTPUT_FA="metavr_baseline_3di.fa"
with open(METAVR_3DI_FA, "r") as infile, open(OUTPUT_FA, "w") as outfile:
    for line in infile:
        if line.startswith(">"):
            outfile.write(line)  # Write header
        else:
            seq_length = len(line.strip())
            baseline_seq = generate_protein_sequence(freq_matrix, seq_length)
            outfile.write(baseline_seq + "\n")

### Calculate pairwise 3Di identity

In [11]:
%%bash
# Generate a TSV file with the following columns:
# [MetaVR ID], [AF3 3Di], [Baseline 3Di]
paste \
	<(awk '{printf "%s\t", $0; getline; print}' "/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/metavr/db/metavr_3di.fa") \
	<(awk '{printf "%s\t", $0; getline; print}' "metavr_baseline_3di.fa" | sort | cut -f2) \
	> "metavr_baseline_comp.tsv"

# Ensure everything has equal length
awk 'length($2) != length($3) {print "Sanity check: ERROR"} END{print "Sanity check: OK"}' \
	"metavr_baseline_comp.tsv"

Sanity check: OK


In [19]:
from pathlib import Path
import os

outfile = open("metavr_baseline_identity.tsv", 'w')
x = []
with open("metavr_baseline_comp.tsv") as f:
	for line in f:
		metaVR_id, seq1, seq2 = line.strip().split('\t')
		metaVR_id = metaVR_id[1:]  # remove '>' from FASTA header

		identity = sum(a == b for a, b in zip(seq1, seq2)) / len(seq1) * 100

		print(f'{metaVR_id}\t{identity:.4f}', file=outfile)
		x.append(identity)

print(f"Average identity: {sum(x)/len(x):.4f}%")

Average identity: 14.7584%
